In [2]:
import nest_asyncio
nest_asyncio.apply()

from clip_client import Client

# Using the IP found in your server logs
c = Client('grpc://172.17.0.8:51000') 

c.profile()

Roundtrip  28ms  100% 
├──  Client-server network  4ms  13% 
└──  Server  24ms  87% 
    ├──  Gateway-CLIP network  2ms  8% 
    └──  CLIP model  22ms  92%

{'Roundtrip': 27.621678998912103,
 'Client-server network': 3.6216789989121025,
 'Server': 24,
 'Gateway-CLIP network': 2,
 'CLIP model': 22}

In [3]:
# Some Markdown?

In [6]:
import os
import glob
from docarray import Document, DocumentArray
# 2. Find all images in the directory
image_dir = os.path.expanduser('~/workspace/Datasets/Pictures/')
# Add any other extensions you have (e.g., .png, .jpeg)
image_paths = glob.glob(os.path.join(image_dir, '*')) 

# Filter to ensure we only try to open images
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
image_paths = [p for p in image_paths if os.path.splitext(p)[1].lower() in valid_extensions]

print(f"Found {len(image_paths)} images in {image_dir}")


📂 Found 758 images in /home/jovyan/workspace/Datasets/Pictures/


In [7]:
docs = DocumentArray([Document(uri=path) for path in image_paths])

def data_generator(docs):
    for doc in docs:
        try:
            # IMPORTANT: This loads the file content so it can be sent 
            # across the container boundary to the server.
            doc.load_uri_to_blob() 
            yield doc
        except Exception as e:
            print(f"Could not load {doc.uri}: {e}")

In [9]:
print(f"Sending {len(docs)} images to CLIP server...")

# batch_size=16 is a safe number to prevent timeouts or OOM on the GPU
embeddings = c.encode(
    data_generator(docs), 
    batch_size=16, 
    show_progress=True
)

print(f"Finished! Embeddings Shape: {embeddings.embeddings.shape}")

Output()

Sending 758 images to CLIP server...


Finished! Embeddings Shape: (758, 512)


In [15]:
import os
from docarray import Document

# 1. Load the Control Image
control_dir = os.path.expanduser('~/workspace/Datasets/Pictures/Control/')
# Grab the first image file found in the Control folder
control_files = [f for f in os.listdir(control_dir) if not f.startswith('.')]

if not control_files:
    raise FileNotFoundError("No image found in the Control folder!")

control_path = os.path.join(control_dir, control_files[0])
print(f" Using Control Image: {control_path}")

# 2. Prepare the Document
d_control = Document(uri=control_path)
d_control.load_uri_to_blob()  # Load content to send to server

# 3. Encode the Control Image
# We wrap it in a list [d_control] because encode expects a list/array
control_vec = c.encode([d_control])

# 4. Find Top 10 Matches
# 'embeddings' is your existing DocumentArray with the 758 images
embeddings.match(control_vec, limit=10)

print(f"\n Top 10 images similar to '{control_files[0]}':")

# The results are stored in the .matches of the query document
# Since we only queried 1 image (control_vec is shape 1xD), we look at embeddings[0]
for i, m in enumerate(embeddings[0].matches, 1):
    filename = os.path.basename(m.uri)
    score = m.scores['cosine'].value
    print(f"   {i}. [{score:.4f}] {filename}")

 Using Control Image: /home/jovyan/workspace/Datasets/Pictures/Control/41727336_001_20fe.jpg

 Top 10 images similar to '41727336_001_20fe.jpg':
   1. [0.4174] 41727336_001_20fe.jpg
